# sld-poc — Stage A: train the chunk codec (JAX / TPU)

Trains the token↔latent codec on a Colab **TPU v5e**.

**Before running:** Runtime → Change runtime type → **TPU**.

Code is cloned fresh from your GitHub repo each session (always matches your latest push); data + checkpoints persist on Drive. If Colab disconnects, re-run the **Train** cell — it auto-resumes.

> **Why we `import codec` instead of `!python codec.py`:** a TPU can only be used by one process. The notebook kernel claims it (Cell 1), so we run the code *in the kernel* — a subprocess (`!python`) would fail with *"TPU already in use"*.

## 1. Confirm the TPU
Expect a `TpuDevice`, backend `tpu`, and a bf16 matmul result.

In [ ]:
import jax, jax.numpy as jnp
print(jax.devices())
print(jax.default_backend())
x = jnp.ones((2048, 2048), jnp.bfloat16)
print((x @ x).sum())

## 2. Mount Google Drive
Click through the authorization popup.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 3. Clone the repo + install deps
Clones your repo fresh to local disk each session (so it always matches your latest `git push`). Make the repo **public**, or you'll need a GitHub token to clone. (jax/jaxlib are preinstalled on the TPU runtime — `requirements.txt` only adds flax/optax/datasets/tiktoken.)

In [ ]:
import os
REPO_URL = 'https://github.com/mccooking/sld-poc.git'
REPO_DIR = '/content/sld-poc'                          # local; re-cloned each session (always fresh)
DATA_DIR = '/content/drive/MyDrive/sld-poc-data'       # data + checkpoints (persist on Drive)
if os.path.exists(REPO_DIR):
    !cd $REPO_DIR && git pull
else:
    !git clone $REPO_URL $REPO_DIR
!pip -q install -r $REPO_DIR/requirements.txt
os.makedirs(DATA_DIR, exist_ok=True)
print('repo:', REPO_DIR, '| data:', DATA_DIR)

## 4. Build the data cache (once, a few minutes)
Streams TinyStories, tokenizes, chunks. Skips itself if the cache exists.

*Already tokenized on a previous run? In a separate cell run `!mkdir -p $DATA_DIR/data && cp /content/drive/MyDrive/clgd/data/tinystories_K4.npy $DATA_DIR/data/` first to reuse it.*

In [ ]:
import sys, importlib
SRC = f'{REPO_DIR}/src'
if SRC not in sys.path: sys.path.insert(0, SRC)
import codec; importlib.reload(codec)
codec.prep(codec.paths(DATA_DIR))

## 5. Train  ⟵  re-run this cell after any disconnect (auto-resumes)
Watch `tok_acc` climb toward ~99.8% (streams live). First step pauses a few seconds to JIT-compile — not a hang. After editing code: `git push` → re-run Cell 3 → re-run this (the `reload` picks up changes).

In [ ]:
import sys, importlib
SRC = f'{REPO_DIR}/src'
if SRC not in sys.path: sys.path.insert(0, SRC)
import codec; importlib.reload(codec)
codec.train(codec.paths(DATA_DIR))

## 6. Round-trip demo
Encodes a sentence to latents and decodes it back; prints how many tokens matched.

In [ ]:
import sys, importlib
SRC = f'{REPO_DIR}/src'
if SRC not in sys.path: sys.path.insert(0, SRC)
import codec; importlib.reload(codec)
codec.demo(codec.paths(DATA_DIR))